# GoMeal ML Rank And Trend Integration

This notebook shows how to update:

- `routes/feed/rank/rank.py`
- `routes/feed/trending/trend.py`

so both feeds use the multi-user learning state created by `brain/api/subscriber.py`.

## Shared Collaborative Boost Helper

Put this helper in a shared file later, for example:

`routes/feed/rank/brain_boosts.py`

For now, you can copy it into both `rank.py` and `trend.py`.

In [ ]:
def _get_collaborative_boost(brain, user_neuron, post_id: int, config) -> float:
    if brain is None or user_neuron is None:
        return 0.0

    candidate_neuron = brain.get_neuron_by_source_id("Recipe", post_id)

    if candidate_neuron is None:
        candidate_neuron = brain.get_neuron_by_source_id("recipe", post_id)

    if candidate_neuron is None:
        candidate_neuron = brain.get_neuron_by_source_id("post", post_id)

    if candidate_neuron is None:
        return 0.0

    viewer_recent_recipe_ids = getattr(
        brain,
        "user_recent_recipe_activations",
        {},
    ).get(user_neuron.neuron_id, [])

    collaborative_strength = 0

    for recent_recipe_id in viewer_recent_recipe_ids:
        if recent_recipe_id == candidate_neuron.neuron_id:
            continue

        pair = tuple(sorted([recent_recipe_id, candidate_neuron.neuron_id]))
        collaborative_strength += getattr(
            brain,
            "recipe_coactivation_counts",
            {},
        ).get(pair, 0)

    if collaborative_strength < config.multi_user_min_coactivation:
        return 0.0

    return collaborative_strength * config.multi_user_boost

## Update `rank.py`

First, preserve `user_neuron` outside the brain block:

```python
brain_ids = set()
user_neuron = None

if _brain:
    user_neuron = _brain.get_neuron_by_source_id("user", int(user_sub))
    ...
```

Then inside the post scoring loop, after direct `brain_boost`, add collaborative boost:

```python
_score = float(np.dot(user_vec, post_vec))

if post_id in brain_ids:
    _score += config.brain_boost

_score += _get_collaborative_boost(
    brain=_brain,
    user_neuron=user_neuron,
    post_id=post_id,
    config=config,
)
```

The final rank score becomes:

```text
semantic similarity
+ direct brain boost
+ multi-user collaborative boost
- seen penalty
```

## Update `trend.py`

Add a brain injection function to `routes/feed/trending/trend.py`:

```python
_brain = None


def _set_trends_brain(brain):
    global _brain
    _brain = brain
```

Then wherever the server boots and calls `_set_rank_brain(brain)` and `_set_subscription_brain(brain)`, also call:

```python
_set_trends_brain(brain)
```

Inside `_post_trends`, after loading `user_vec`, resolve the viewer neuron:

```python
user_neuron = None

if _brain is not None:
    user_neuron = _brain.get_neuron_by_source_id("user", int(user_sub))
```

Then inside the scoring loop, after personalized vector boost, add:

```python
if user_vec is not None:
    personalized_boost = float(np.dot(user_vec, post_vec))
    _score += personalized_boost * 0.10

_score += _get_collaborative_boost(
    brain=_brain,
    user_neuron=user_neuron,
    post_id=post_id,
    config=config,
)
```

The final trend score becomes:

```text
weighted action count
* recency decay
+ light personalized boost
+ multi-user collaborative boost
- seen penalty
```

## Why This Works

The collaborative boost is viewer-aware.

It does not boost every popular coactivated post for everyone. It only boosts a candidate when it is connected to recipes the viewer has recently activated.

```text
viewer recent recipe A
candidate recipe B
recipe_coactivation_counts[(A, B)] >= threshold
=> boost B
```

In [ ]:
class Config:
    multi_user_min_coactivation = 3
    multi_user_boost = 0.12


class Node:
    def __init__(self, neuron_id):
        self.neuron_id = neuron_id


class ToyBrain:
    def __init__(self):
        self.user = Node(1)
        self.recipe_by_post = {101: Node(10), 202: Node(20), 303: Node(30)}
        self.user_recent_recipe_activations = {1: [10]}
        self.recipe_coactivation_counts = {(10, 20): 3, (10, 30): 1}

    def get_neuron_by_source_id(self, source_model, source_id):
        if source_model == "user" and source_id == 1:
            return self.user
        if source_model == "Recipe":
            return self.recipe_by_post.get(source_id)
        return None


toy_brain = ToyBrain()
viewer = toy_brain.get_neuron_by_source_id("user", 1)

for post_id in [101, 202, 303]:
    print(post_id, _get_collaborative_boost(toy_brain, viewer, post_id, Config()))